# INJECTION-R3: RSG Boost Pass

**Stage:** R-3 of the INJECTION pipeline  
**Input:** `r2_best_weights.h5` (joint fine-tuned from R2)  
**Output:** `r3_best_weights.h5` (RSG-boosted)

### What this stage does
- Loads R2 weights and **freezes everything except `rsg_*` layers**
- Trains RSG head in isolation with boosted loss weight (0.60)
- **DriftGuard**: halts if ATS val MAE exceeds 8.5 (0–100 scale)
- Early stopping on RSG val accuracy (patience = 6)

### Target Gates
| Metric | Target | Hard Ceiling |
|--------|--------|--------------|
| RSG val Accuracy | > 0.62 | — |
| ATS val MAE (0–100) | — | ≤ 8.5 |
| Domain val F1 | — | ≥ 0.78 |

### Required files — upload to your `ATS_R3/` folder on Google Drive
```
MyDrive/ATS_R3/
  ats_tokenized.npz        <- from data/tokenized/
  rsg_tokenized.npz        <- from data/tokenized/
  r2_best_weights.h5       <- from model/unified_model/new_unified_model/
  data_splits.json         <- from model/unified_model/
```
Output (`r3_best_weights.h5`, `r3_training_log.csv`) auto-saved to `ATS_R3/output/`.

> **Runtime:** Set to **T4 GPU** via `Runtime -> Change runtime type -> T4 GPU`

In [ ]:
# Cell 1: GPU check
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print('No GPU detected!')
    print('Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU')
    print('Then re-run all cells.')
else:
    print(result.stdout)

In [ ]:
# Cell 2: Install dependencies
# Pin to 4.44.2 — TFAutoModel was removed from transformers in 4.47+.
# The [tf] extra alone does NOT help on versions >= 4.47.
!pip install -q "transformers==4.44.2" scikit-learn
print('Dependencies installed.')
print()
print('>>> ACTION REQUIRED: Runtime -> Restart session, then re-run ALL cells from Cell 1. <<<')

In [ ]:
# Cell 3: Mount Google Drive & configure paths
from google.colab import drive
drive.mount('/content/drive')

# ================================================================
#  CONFIGURE: set this to your ATS_R3 folder on Drive
# ================================================================
DRIVE_ROOT = '/content/drive/MyDrive/ATS_R3'

import os
from pathlib import Path

DATA_DIR      = Path(DRIVE_ROOT)
R2_WEIGHTS    = str(Path(DRIVE_ROOT) / 'r2_best_weights.h5')
SPLITS_JSON   = str(Path(DRIVE_ROOT) / 'data_splits.json')

OUT_DIR       = Path(DRIVE_ROOT) / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)
R3_BEST       = str(OUT_DIR / 'r3_best_weights.h5')
R3_LOG        = str(OUT_DIR / 'r3_training_log.csv')

print(f'Drive root : {DRIVE_ROOT}')
print(f'R2 weights : {R2_WEIGHTS}')
print(f'Splits     : {SPLITS_JSON}')
print(f'R3 output  : {OUT_DIR}')

In [ ]:
# Cell 4: Verify required files
required = [
    str(DATA_DIR / 'ats_tokenized.npz'),
    str(DATA_DIR / 'rsg_tokenized.npz'),
    R2_WEIGHTS,
    SPLITS_JSON,
]
all_ok = True
for f in required:
    exists = os.path.exists(f)
    size   = f'{os.path.getsize(f) / 1e6:.1f} MB' if exists else 'MISSING'
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status}]  {size:>10}   {f}')
    all_ok = all_ok and exists

if not all_ok:
    raise FileNotFoundError('One or more required files are missing. Upload them to Drive first.')
print('\nAll required files found -- ready to train.')

In [ ]:
# Cell 5: Imports & environment
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import csv, json, sys
import numpy as np
import tensorflow as tf
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
tf.get_logger().setLevel('ERROR')
from sklearn.metrics import f1_score

print(f'TensorFlow : {tf.__version__}')
print(f'GPU devices: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# Cell 6: Hyperparameters
MINILM_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
SEQ_LEN           = 128

LR            = 5e-4
BATCH_SIZE    = 32
MAX_EPOCHS    = 25
PATIENCE      = 6
W_RSG         = 0.60   # boosted RSG loss weight

ATS_MAE_CEILING = 8.5
RSG_TARGET_ACC  = 0.62
DOMAIN_F1_FLOOR = 0.78

print('Hyperparameters configured.')
print(f'  LR={LR}  BATCH={BATCH_SIZE}  MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}')
print(f'  RSG loss weight (boosted): {W_RSG}')
print(f'  DriftGuard ceiling: ATS MAE <= {ATS_MAE_CEILING}')

In [ ]:
# Cell 7: MiniLM model definition (self-contained)
import transformers as _tf_check
_major, _minor = (int(x) for x in _tf_check.__version__.split('.')[:2])
if (_major, _minor) >= (4, 47):
    raise RuntimeError(
        f'transformers {_tf_check.__version__} is loaded — TFAutoModel was removed in 4.47.\n'
        'Fix: run Cell 2, then go to Runtime -> Restart session, then re-run all cells from Cell 1.'
    )

from transformers import TFAutoModel

def mean_pool_l2(last_hidden, attention_mask):
    mask    = tf.cast(tf.expand_dims(attention_mask, -1), tf.float32)
    sum_emb = tf.reduce_sum(last_hidden * mask, axis=1)
    count   = tf.maximum(tf.reduce_sum(mask, axis=1), 1e-9)
    return tf.nn.l2_normalize(sum_emb / count, axis=-1)


class MiniLMEncoderLayer(tf.keras.layers.Layer):
    def __init__(self, bert_model, **kwargs):
        kwargs.setdefault('trainable', False)
        super().__init__(**kwargs)
        self._bert = bert_model
        self._bert.trainable = False

    def call(self, inputs, training=False):
        input_ids, attention_mask = inputs
        out = self._bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=tf.zeros_like(input_ids),
            training=False,
        )
        return mean_pool_l2(out.last_hidden_state, attention_mask)

    def get_config(self):
        return super().get_config()


def build_unified_minilm_model(bert_model):
    resume_ids  = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='resume_input_ids')
    resume_mask = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='resume_attention_mask')
    jd_ids      = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='jd_input_ids')
    jd_mask     = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='jd_attention_mask')

    encoder    = MiniLMEncoderLayer(bert_model, name='minilm_encoder')
    resume_emb = encoder([resume_ids,  resume_mask])
    jd_emb     = encoder([jd_ids,      jd_mask])

    cosine_sim = tf.keras.layers.Dot(axes=1, normalize=True,  name='cosine_sim')([resume_emb, jd_emb])
    dot_prod   = tf.keras.layers.Dot(axes=1, normalize=False, name='dot_prod')  ([resume_emb, jd_emb])
    ats_feat   = tf.keras.layers.Concatenate(name='ats_features')([resume_emb, jd_emb, cosine_sim, dot_prod])

    x1        = tf.keras.layers.Dense(256, activation='relu',    name='ats_dense1')(ats_feat)
    x1        = tf.keras.layers.Dropout(0.3,                     name='ats_drop1')(x1)
    x1        = tf.keras.layers.Dense(64,  activation='relu',    name='ats_dense2')(x1)
    x1        = tf.keras.layers.Dropout(0.2,                     name='ats_drop2')(x1)
    ats_score = tf.keras.layers.Dense(1, activation='sigmoid',   name='ats_score')(x1)

    x2           = tf.keras.layers.Dense(256, activation='relu',  name='dom_dense1')(jd_emb)
    x2           = tf.keras.layers.Dropout(0.3,                   name='dom_drop1')(x2)
    x2           = tf.keras.layers.Dense(128, activation='relu',  name='dom_dense2')(x2)
    x2           = tf.keras.layers.Dropout(0.2,                   name='dom_drop2')(x2)
    domain_probs = tf.keras.layers.Dense(7, activation='softmax', name='domain_probs')(x2)

    x3           = tf.keras.layers.Dense(512, activation='relu',  name='rsg_dense1')(resume_emb)
    x3           = tf.keras.layers.BatchNormalization(            name='rsg_bn1')(x3)
    x3           = tf.keras.layers.Dropout(0.4,                   name='rsg_drop1')(x3)
    x3           = tf.keras.layers.Dense(256, activation='relu',  name='rsg_dense2')(x3)
    x3           = tf.keras.layers.BatchNormalization(            name='rsg_bn2')(x3)
    x3           = tf.keras.layers.Dropout(0.3,                   name='rsg_drop2')(x3)
    x3           = tf.keras.layers.Dense(128, activation='relu',  name='rsg_dense3')(x3)
    x3           = tf.keras.layers.BatchNormalization(            name='rsg_bn3')(x3)
    x3           = tf.keras.layers.Dropout(0.3,                   name='rsg_drop3')(x3)
    rsg_template = tf.keras.layers.Dense(46, activation='softmax', name='rsg_template')(x3)

    return tf.keras.Model(
        inputs=[resume_ids, resume_mask, jd_ids, jd_mask],
        outputs=[ats_score, domain_probs, rsg_template],
        name='unified_ats_minilm',
    )

print(f'transformers {_tf_check.__version__} -- TFAutoModel available.')
print('Model definition ready.')

In [ ]:
# Cell 8: Load tokenized data & splits
print('Loading tokenized data...')
ats_data = np.load(str(DATA_DIR / 'ats_tokenized.npz'))
rsg_data = np.load(str(DATA_DIR / 'rsg_tokenized.npz'))

with open(SPLITS_JSON) as f:
    splits = json.load(f)

ats_vl_idx = np.array(splits['ats_val'])
rsg_tr_idx = np.array(splits['rsg_train'])
rsg_vl_idx = np.array(splits['rsg_val'])

ATS_KEYS = ('resume_input_ids', 'resume_attention_mask',
             'jd_input_ids', 'jd_attention_mask', 'ats_scores', 'domain_labels')
RSG_KEYS = ('profile_input_ids', 'profile_attention_mask', 'rsg_labels')

ats_vl = {k: ats_data[k][ats_vl_idx] for k in ATS_KEYS}
rsg_tr = {k: rsg_data[k][rsg_tr_idx] for k in RSG_KEYS}
rsg_vl = {k: rsg_data[k][rsg_vl_idx] for k in RSG_KEYS}

n_rsg_tr = len(rsg_tr_idx)
print(f'  ATS  val ={len(ats_vl_idx):,}  (drift guard only)')
print(f'  RSG  train={n_rsg_tr:,}  val={len(rsg_vl_idx):,}')

In [ ]:
# Cell 9: Build model & load R2 weights
print(f'Loading MiniLM encoder ({MINILM_MODEL_NAME})...')
bert_model = TFAutoModel.from_pretrained(MINILM_MODEL_NAME, from_pt=True)
bert_model.trainable = False

model = build_unified_minilm_model(bert_model)
model.load_weights(R2_WEIGHTS)
print(f'R2 weights loaded: {R2_WEIGHTS}')

In [ ]:
# Cell 10: Freeze all layers; unfreeze only rsg_* layers
for layer in model.layers:
    layer.trainable = False

rsg_layer_names = []
for layer in model.layers:
    if layer.name.startswith('rsg_'):
        layer.trainable = True
        rsg_layer_names.append(layer.name)

trainable = sum(int(np.prod(v.shape)) for v in model.trainable_weights)
frozen    = sum(int(np.prod(v.shape)) for v in model.non_trainable_weights)
print(f'Freeze strategy applied.')
print(f'  Trainable layers : {rsg_layer_names}')
print(f'  Trainable params : {trainable:,}')
print(f'  Frozen params    : {frozen:,}')

In [ ]:
# Cell 11: Eval helpers + measure R2 baseline (before any R3 gradient)
def eval_ats(data):
    n = len(data['ats_scores'])
    ats_preds, dom_preds = [], []
    for s in range(0, n, BATCH_SIZE):
        e = min(s + BATCH_SIZE, n)
        ap, dp, _ = model(
            [data['resume_input_ids'][s:e], data['resume_attention_mask'][s:e],
             data['jd_input_ids'][s:e],     data['jd_attention_mask'][s:e]],
            training=False,
        )
        ats_preds.append(ap.numpy())
        dom_preds.append(dp.numpy())
    ats_preds = np.concatenate(ats_preds).squeeze(-1)
    dom_preds = np.concatenate(dom_preds)
    mae_100  = float(np.mean(np.abs(ats_preds - data['ats_scores']))) * 100.0
    dom_true = data['domain_labels']
    dom_f1   = f1_score(dom_true, np.argmax(dom_preds, 1), average='macro', zero_division=0)
    return mae_100, dom_f1


def eval_rsg(data):
    m = len(data['rsg_labels'])
    rsg_preds = []
    for s in range(0, m, BATCH_SIZE):
        e    = min(s + BATCH_SIZE, m)
        pids = data['profile_input_ids'][s:e]
        pmsk = data['profile_attention_mask'][s:e]
        _, _, rp = model([pids, pmsk, pids, pmsk], training=False)
        rsg_preds.append(rp.numpy())
    rsg_preds = np.concatenate(rsg_preds)
    acc = float(np.mean(np.argmax(rsg_preds, 1) == data['rsg_labels']))
    ce  = float(np.mean(
        tf.keras.losses.sparse_categorical_crossentropy(data['rsg_labels'], rsg_preds).numpy()
    ))
    return acc, ce


print('Measuring R2 baseline...')
r2_val_mae, r2_dom_f1 = eval_ats(ats_vl)
r2_rsg_acc, _         = eval_rsg(rsg_vl)
print(f'  R2 val ATS MAE : {r2_val_mae:.2f}')
print(f'  R2 val Dom F1  : {r2_dom_f1:.4f}')
print(f'  R2 val RSG Acc : {r2_rsg_acc:.4f}')

In [ ]:
# Cell 12: Optimizer & train step
optimizer = tf.keras.optimizers.Adam(learning_rate=LR)
_sce      = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)


def train_step_rsg(p_ids, p_mask, rsg_y):
    with tf.GradientTape() as tape:
        _, _, rsg_p = model([p_ids, p_mask, p_ids, p_mask], training=True)
        loss = W_RSG * _sce(rsg_y, rsg_p)
    optimizer.apply_gradients(
        zip(tape.gradient(loss, model.trainable_variables), model.trainable_variables)
    )
    return float(loss)


print('Optimizer and train step ready.')
print(f'  Adam(lr={LR})  |  only rsg_* variables will receive gradients')

In [ ]:
# Cell 13: Training loop (RSG-only, with DriftGuard)
print('=' * 60)
print(f'  INJECTION-R3 Training  (max={MAX_EPOCHS} epochs, patience={PATIENCE})')
print('=' * 60)

rng              = np.random.default_rng(42)
best_rsg_acc     = -1.0
patience_counter = 0
best_epoch       = -1
log_rows         = []
drift_triggered  = False

batches_per_epoch = -(-n_rsg_tr // BATCH_SIZE)
print(f'  RSG batches/epoch: {batches_per_epoch}\n')

for epoch in range(MAX_EPOCHS):
    perm       = rng.permutation(n_rsg_tr)
    batch_idxs = [perm[i:i + BATCH_SIZE] for i in range(0, n_rsg_tr, BATCH_SIZE)]

    epoch_loss = 0.0
    for bidx in batch_idxs:
        epoch_loss += train_step_rsg(
            rsg_tr['profile_input_ids'][bidx],
            rsg_tr['profile_attention_mask'][bidx],
            rsg_tr['rsg_labels'][bidx],
        )
    train_loss = epoch_loss / len(batch_idxs)

    # NaN hard stop
    if np.isnan(train_loss):
        print(f'\nHARD STOP -- NaN train_loss at epoch {epoch + 1}')
        break

    # Validation
    val_rsg_acc, _ = eval_rsg(rsg_vl)
    val_ats_mae, val_dom_f1 = eval_ats(ats_vl)

    print(
        f'Epoch {epoch+1:3d}/{MAX_EPOCHS}  '
        f'train={train_loss:.4f}  '
        f'RSG_Acc={val_rsg_acc:.4f}  ATS_MAE={val_ats_mae:.2f}  Dom_F1={val_dom_f1:.4f}'
    )

    log_rows.append({
        'epoch':       epoch + 1,
        'train_loss':  round(train_loss,   6),
        'val_rsg_acc': round(val_rsg_acc,  4),
        'val_ats_mae': round(val_ats_mae,  4),
        'val_dom_f1':  round(val_dom_f1,   4),
    })

    # DriftGuard
    if val_ats_mae > ATS_MAE_CEILING:
        print(f'\n*** DRIFT GUARD -- HARD STOP at epoch {epoch + 1}: '
              f'ATS MAE {val_ats_mae:.2f} > {ATS_MAE_CEILING} ceiling. '
              'Restoring best checkpoint. ***')
        if best_epoch >= 0:
            model.load_weights(R3_BEST)
        drift_triggered = True
        break

    # Checkpoint on best RSG val accuracy
    if val_rsg_acc > best_rsg_acc:
        best_rsg_acc     = val_rsg_acc
        best_epoch       = epoch + 1
        patience_counter = 0
        model.save_weights(R3_BEST)
        print(f'  [Saved] RSG_Acc={val_rsg_acc:.4f}  ATS_MAE={val_ats_mae:.2f}  -> epoch {epoch + 1}')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping (patience={PATIENCE}) at epoch {epoch + 1}')
            break

print('\nTraining loop complete.')
if drift_triggered:
    print('NOTE: DriftGuard triggered -- best pre-drift weights restored.')

In [ ]:
# Cell 14: Save training log
if log_rows:
    with open(R3_LOG, 'w', newline='') as fh:
        writer = csv.DictWriter(fh, fieldnames=log_rows[0].keys())
        writer.writeheader()
        writer.writerows(log_rows)
    print(f'Training log saved -> {R3_LOG}')

In [ ]:
# Cell 15: Final evaluation & gate report
print('Loading best checkpoint and running final evaluation...')
if best_epoch >= 0:
    model.load_weights(R3_BEST)

final_rsg_acc, _ = eval_rsg(rsg_vl)
final_ats_mae, final_dom_f1 = eval_ats(ats_vl)

ats_pass    = final_ats_mae <= ATS_MAE_CEILING
rsg_pass    = final_rsg_acc >= RSG_TARGET_ACC
domain_pass = final_dom_f1  >= DOMAIN_F1_FLOOR

print('\n' + '=' * 60)
print('  INJECTION-R3 GATE RESULTS')
print('=' * 60)
print(f'  RSG val accuracy   : {final_rsg_acc:.4f}  '
      f'{"PASS" if rsg_pass    else "FAIL"}  (target > {RSG_TARGET_ACC})')
print(f'  ATS val MAE (0-100): {final_ats_mae:.3f}  '
      f'{"PASS" if ats_pass    else "FAIL"}  (ceiling <= {ATS_MAE_CEILING})')
print(f'  Domain F1 (macro)  : {final_dom_f1:.4f}  '
      f'{"PASS" if domain_pass else "FAIL"}  (floor >= {DOMAIN_F1_FLOOR})')
print(f'\n  R2 baseline RSG Acc: {r2_rsg_acc:.4f}  '
      f'->  delta: {final_rsg_acc - r2_rsg_acc:+.4f}')
print(f'  Best epoch         : {best_epoch}')
print(f'  Output             : {R3_BEST}')
print('=' * 60)

if ats_pass and rsg_pass and domain_pass:
    print('\nR3 COMPLETE -- proceed to R4')
else:
    print('\nR3 INCOMPLETE -- one or more gates failed. Review metrics above.')

In [ ]:
# Cell 16: Download results to local machine
# Weights are already on Google Drive (persistent).
# Use this cell to also download directly to your browser.
from google.colab import files

print('Downloading r3_best_weights.h5 ...')
files.download(R3_BEST)

print('Downloading r3_training_log.csv ...')
files.download(R3_LOG)

print('Save r3_best_weights.h5 to:')
print('  model/unified_model/r3_best_weights.h5')